# Data Understanding: Orders Dataset

## Purpose

This notebook begins the data-understanding phase for the E-Commerce Delivery Intelligence project. We start with the orders dataset because it is the central table: each row represents an order and contains the purchase timestamp, delivery estimate, order status, and delivery timestamps needed later to define the classification and regression targets.

The notebook currently covers three steps on the `orders` DataFrame: (1) loading and previewing the source CSV, (2) structural inspection of its shape, column names, and data types, and (3) uniqueness and duplicate inspection of `order_id` and complete rows. It only inspects the data; it does not clean, transform, or join it.

## 1. Load the Orders Data

`Path.cwd()` gets the directory from which the notebook is running. Because this notebook is intended to run from `notebooks/`, its parent is the project root. Building the path with `pathlib` keeps it portable and avoids a machine-specific absolute path.

The existence check raises a clear error containing the expected location if the CSV is missing. `pd.read_csv(...)` then reads the original file without modifying it.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent
orders_path = project_root / "data" / "raw" / "olist_orders_dataset.csv"

if not orders_path.exists():
    raise FileNotFoundError(f"Orders dataset not found at expected path: {orders_path}")

orders = pd.read_csv(orders_path)

### 1.1 Preview the Loaded Orders

`orders.head()` displays the first five rows by default. This small preview confirms that the DataFrame loaded and lets us see the column layout without starting data-quality analysis or other EDA tasks.

In [2]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


## 2. Structural Inspection

Structural inspection helps us understand the dataset's size, the information available in its columns, and how Pandas currently represents that information. Establishing this structure gives us a clear foundation before any deeper analysis.

### 2.1 Dataset Shape

`DataFrame.shape` returns a tuple in the form `(number_of_rows, number_of_columns)`. Each row represents one order record, while each column represents a recorded attribute about that order. Dataset dimensions matter in ML projects because they establish how many observations and variables are available before later validation and modeling decisions.

In [3]:
orders.shape

(99441, 8)

### 2.2 Column Names

`DataFrame.columns` contains the labels of all available columns. Reviewing these names before feature engineering helps us recognize identifiers, possible predictors, and columns related to future target construction. Calling `.tolist()` presents the labels as a simple Python list.

In [4]:
orders.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

### 2.3 Column Data Types

`DataFrame.dtypes` reports the data type Pandas assigned to each column. Data types determine which calculations and transformations are valid and later influence preprocessing choices for ML. Because this CSV was loaded without explicit date parsing, timestamp columns may initially appear as `object` (or `str` in newer Pandas versions), meaning Pandas currently treats their values as strings rather than datetime objects.

In [5]:
orders.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

### 2.4 Learning Questions

1. What does one row of the orders dataset represent?
2. How many rows and columns are present?
3. Which columns are identifiers?
4. Which columns represent dates or timestamps?
5. Are the timestamp columns currently stored as datetime objects or strings?
6. Why is knowing the data types important before performing calculations?

## 3. Uniqueness and Duplicate Inspection

Uniqueness means that a key identifies only one record in a relational dataset. Because the orders table is expected to contain one row per order, `order_id` should uniquely identify each row. Repeated identifiers can represent more than one record for the same order, while fully duplicated rows are records whose values match across every column. Either issue can distort counts, summaries, and ML training by giving some observations extra weight. We inspect both forms first so their cause and meaning can be understood before any decision about removal.

### 3.1 Count Unique Order IDs

`nunique()` counts the distinct values in a Series. By default, it excludes missing values from the count.

In [6]:
orders["order_id"].nunique()

99441

### 3.2 Verify Order ID Uniqueness

`is_unique` checks whether every value in a Series occurs only once. It is a property, so it is used without parentheses. `True` means no value is repeated; `False` means at least one value is repeated.

In [7]:
orders["order_id"].is_unique

True

### 3.3 Count Duplicated Order IDs

`duplicated()` returns a Boolean Series that marks repeated occurrences. By default, the first occurrence is not marked as duplicated. Calling `.sum()` counts the `True` values, so the result measures repeated occurrences rather than the number of distinct IDs that repeat. This inspection does not delete or filter any rows. The result is displayed as `np.int64(0)` because Pandas returns a NumPy integer; it simply means the count is 0.

In [8]:
orders["order_id"].duplicated().sum()

np.int64(0)

### 3.4 Count Fully Duplicated Rows

A duplicated identifier repeats the key even when other column values differ. A fully duplicated row matches an earlier row across every column. `orders.duplicated()` checks complete rows, and `.sum()` counts the repeated occurrences without changing the DataFrame. The result is displayed as `np.int64(0)` because Pandas returns a NumPy integer; it simply means the count is 0.

In [9]:
orders.duplicated().sum()

np.int64(0)

### 3.5 Learning Questions

1. What is the difference between the total number of rows and the number of unique order IDs?
2. Why should `order_id` be unique in the orders table?
3. What is the difference between a duplicated order ID and a completely duplicated row?
4. Why might `order_id` legitimately appear multiple times in the `order_items` table?
5. How could accidental duplicate records affect ML model training?
6. Why should we investigate duplicate records before removing them?